# 02 · Newton, convergência e fractais

**Objetivo pedagógico:** Distinguir convergência, estagnação e derivada pequena e interpretar bacias de atração.

**Luigi Scofano de Araujo · USP 13680334**

Reelaboração de estudo de um tema presente no acervo recuperado de 2025. Esta edição foi preparada em 2026. Não é uma cópia autenticada de uma entrega nem uma afirmação de conclusão das questões originais. Os arquivos recuperados estão em `acervo/`, com créditos e limitações em `docs/PROCEDENCIA.md`.

Execute as células em ordem, em kernel novo. Os dados são locais/sintéticos e as conclusões numéricas são verificadas por asserções; gráficos sozinhos não certificam convergência.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd()/"numericos").is_dir() else Path.cwd().parent
if not (ROOT/"numericos").is_dir():
    raise RuntimeError("Execute a partir da raiz do repositório ou de notebooks/.")
sys.path.insert(0, str(ROOT))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=7, suppress=True)
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True})
FIGURAS = ROOT/"assets"/"figuras"
FIGURAS.mkdir(parents=True, exist_ok=True)
def mostrar(nome):
    plt.tight_layout()
    plt.savefig(FIGURAS/(nome+".svg"), bbox_inches="tight", metadata={"Date": None})
    plt.show()
    plt.close("all")

## 1. Iteração de Newton
A iteração é $x_{k+1}=x_k-f(x_k)/f'(x_k)$. Derivada nula impede o passo; uma sequência que para de mudar não prova que encontrou uma raiz. O motor registra iteradas, resíduos e motivo de parada.

In [2]:
from numericos.raizes import newton, bacias_newton
r = newton(lambda x:x*x-2, lambda x:2*x, 1.)
print("raiz:",r.raiz,"convergiu:",r.convergiu,"motivo:",r.motivo)
print("iterada       resíduo")
for xk, residual in zip(r.iteradas,r.residuos):
    print(f"{xk:14.10f} {residual:.3e}")
assert r.convergiu and abs(r.raiz-np.sqrt(2)) < 1e-10
plt.semilogy(np.maximum(r.residuos,1e-16),"o-")
plt.xlabel("iteração"); plt.ylabel("|f(x)|")
mostrar("02_convergencia")

raiz: 1.4142135623746899 convergiu: True motivo: resíduo abaixo da tolerância
iterada       resíduo
  1.0000000000 1.000e+00
  1.5000000000 2.500e-01
  1.4166666667 6.944e-03
  1.4142156863 6.007e-06
  1.4142135624 4.511e-12


## 2. Raiz múltipla e falha de convergência
Para $f(x)=(x-1)^2$, Newton converge apenas linearmente sob as hipóteses usuais. O resíduo é o quadrado do erro: resíduo pequeno pode coexistir com erro de posição maior. O ciclo de Newton para $x^3-2x+2$, iniciado em zero, alterna entre 0 e 1.

In [3]:
multiple = newton(lambda x:(x-1)**2,lambda x:2*(x-1),2,tol=1e-12)
cycle = newton(lambda x:x**3-2*x+2,lambda x:3*x*x-2,0,maxiter=12)
zero_derivative = newton(lambda x:x*x+1,lambda x:2*x,0)
print("Raiz múltipla: erro",abs(multiple.raiz-1),"resíduo",multiple.residuos[-1])
print("Ciclo:",cycle.iteradas,"/",cycle.motivo)
print("Derivada nula:",zero_derivative.motivo)
assert not cycle.convergiu and not zero_derivative.convergiu
assert np.array_equal(cycle.iteradas[:4],[0,1,0,1])

Raiz múltipla: erro 9.5367431640625e-07 resíduo 9.094947017729282e-13
Ciclo: [0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0.] / máximo de iterações
Derivada nula: derivada pequena ou não finita


## 3. Bacias para $z^3-1$
Cada cor indica a raiz cujo resíduo ficou abaixo da tolerância. O rótulo −1 é reservado a pontos não certificados no limite de iterações, inclusive a origem onde a derivada é nula. A figura depende da malha e dos critérios; fronteiras detalhadas não são prova de caos de um circuito.

In [4]:
from matplotlib.colors import ListedColormap
axis,labels = bacias_newton(tamanho=241,maxiter=50)
plt.imshow(labels,origin="lower",extent=[axis[0],axis[-1],axis[0],axis[-1]],
           cmap=ListedColormap(["#182334","#67d5df","#b998ff","#80d6a5"]),vmin=-1,vmax=2)
plt.xlabel("Re(z₀)");plt.ylabel("Im(z₀)")
print("Pontos sem convergência certificada:",np.count_nonzero(labels==-1))
assert set(np.unique(labels)) == {-1,0,1,2}
mostrar("02_bacias")

Pontos sem convergência certificada: 1


## Interpretação e exercício
A convergência depende do chute e da função. A derivada mínima usada é um limiar absoluto didático e pode exigir reescala do problema. Compare resultados para três malhas e dois limites de iterações; mantenha a cor dos pontos sem convergência explícita.

**Evidência esperada:** tabela de resíduos, diagnóstico do ciclo e justificativa para o rótulo −1.

## Figuras desta execução

![02_bacias](../assets/figuras/02_bacias.svg)

![02_convergencia](../assets/figuras/02_convergencia.svg)